# Module 2 · Chapter 2 — 조건부 확률과 베이즈 정리

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 조건부 확률 정의, 베이즈 정리, 기저율 무시 함정
- **이 노트북**: 유방암 검진 시나리오로 베이즈 갱신을 직접 계산하고 시각화

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 시나리오 데이터 — 유방암 검진

- 유병률: 1% (P(암) = 0.01)
- 민감도: 90% (P(양성|암) = 0.90)
- 특이도: 91% (P(음성|정상) = 0.91) → 위양성률 = 9%

In [ ]:
# 1000명 자연수 단위 계산
n = 1000
prevalence  = 0.01
sensitivity = 0.90
specificity = 0.91
fpr = 1 - specificity  # 위양성률

n_cancer  = int(n * prevalence)         # 10명
n_normal  = n - n_cancer                 # 990명
tp = round(n_cancer * sensitivity)       # 9명
fn = n_cancer - tp                       # 1명
fp = round(n_normal * fpr)               # 89명
tn = n_normal - fp                       # 901명

cm = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=["실제 암", "실제 정상"],
    columns=["검사 양성", "검사 음성"]
)
print("=== 혼동 행렬 (1000명 기준) ===")
print(cm)
print()

ppv = tp / (tp + fp)
print(f"전체 양성: {tp + fp}명 (TP {tp} + FP {fp})")
print(f"양성 중 실제 암: {tp}명")
print(f"PPV = {tp}/{tp+fp} = {ppv:.4f} ({ppv*100:.1f}%)")

## 2. 베이즈 정리를 수식 그대로 코드로

$$P(\text{암}|\text{양성}) = \frac{P(\text{양성}|\text{암}) \cdot P(\text{암})}{P(\text{양성})}$$

In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    """P(A|B) = P(B|A)*P(A) / P(B)"""
    p_pos_given_cancer  = likelihood           # P(양성|암)
    p_pos_given_normal  = false_positive_rate  # P(양성|정상)
    p_cancer            = prior
    p_normal            = 1 - prior

    p_positive = (p_pos_given_cancer * p_cancer +
                  p_pos_given_normal * p_normal)   # 전체 확률 법칙

    posterior = (p_pos_given_cancer * p_cancer) / p_positive
    return posterior, p_positive

posterior, p_pos = bayes(prevalence, sensitivity, fpr)

print(f"P(양성|암)  = {sensitivity}")
print(f"P(암)       = {prevalence}")
print(f"P(양성)     = {p_pos:.4f}")
print()
print(f"P(암|양성)  = {sensitivity}×{prevalence} / {p_pos:.4f}")
print(f"           = {sensitivity*prevalence:.4f} / {p_pos:.4f}")
print(f"           = {posterior:.4f} ({posterior*100:.1f}%)")
print()
print(f"직감(90%)과 실제({posterior*100:.1f}%)의 차이 — 기저율 무시의 결과")

## 3. 유병률을 바꾸면 PPV가 어떻게 달라지는가

In [ ]:
prevalences = np.linspace(0.001, 0.30, 200)
ppv_vals = [bayes(p, sensitivity, fpr)[0] for p in prevalences]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prevalences * 100, [v * 100 for v in ppv_vals],
        color="steelblue", linewidth=2, label="PPV")

for prev_mark, color in [(0.01, "tomato"), (0.05, "orange"), (0.20, "mediumseagreen")]:
    ppv_mark = bayes(prev_mark, sensitivity, fpr)[0]
    ax.scatter(prev_mark*100, ppv_mark*100, color=color, s=100, zorder=5)
    ax.annotate(f"유병률 {prev_mark*100:.0f}%\nPPV {ppv_mark*100:.0f}%",
                xy=(prev_mark*100, ppv_mark*100), xytext=(prev_mark*100+1, ppv_mark*100-8),
                fontsize=9, color=color)

ax.set_xlabel("유병률 (%)")
ax.set_ylabel("양성 예측도 PPV (%)")
ax.set_title("유병률(기저율)에 따른 양성 예측도 변화")
plt.tight_layout()
plt.show()

## 4. 베이즈 순차 갱신 — 검사를 두 번 하면?

첫 번째 검사 양성 → 사후 확률이 새로운 사전 확률이 됩니다.
두 번째 독립적인 검사에서 다시 양성 → 확률이 얼마나 올라가는지 봅니다.

In [ ]:
# 순차적 갱신 (검사 결과가 모두 양성일 때)
prior = prevalence
updates = []
for test_num in range(1, 6):
    posterior_val, _ = bayes(prior, sensitivity, fpr)
    updates.append({"검사 횟수": test_num, "사후 확률": posterior_val})
    prior = posterior_val  # 사후 확률 → 다음 번 사전 확률

update_df = pd.DataFrame(updates)
print("=== 베이즈 순차 갱신 (매번 양성) ===")
print(update_df.to_string(index=False, float_format="{:.4f}".format))
print()
print("주의: 같은 키트를 반복 쓰는 것은 진짜 독립이 아닐 수 있습니다.")
print("오차가 상관되어 있다면 이 갱신은 지나치게 낙관적입니다.")

## 5. 직접 답해 보기

1. P(양성|암)과 P(암|양성)이 왜 다른 숫자인지 이 시나리오로 설명한다면?
2. 유병률이 매우 낮은 질환에서 높은 민감도의 검사를 쓸 때 어떤 문제가 생길 수 있을까?
3. 베이즈 갱신에서 "사전 확률을 어디서 가져올 것인가"가 왜 중요한 문제인가?